# 01 — Scraper

Scrapes the Philippine Supreme Court e-library and downloads all Philippine Reports PDF volumes.

**What this notebook does:**
1. Loads configuration from `.env`
2. Fetches the e-library page and parses PDF links
3. Downloads each PDF with polite delays
4. Logs all activity to `download.log`

In [ ]:
import os
import re
import time
import logging
from pathlib import Path

import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from tqdm.notebook import tqdm

## Configuration

In [ ]:
load_dotenv()

DOWNLOAD_PATH = Path(os.getenv("DOWNLOAD_PATH", "./downloads"))
REQUEST_DELAY = float(os.getenv("REQUEST_DELAY", "1"))
BASE_URL = "https://elibrary.judiciary.gov.ph"
LISTING_URL = f"{BASE_URL}/philippinereports"

DOWNLOAD_PATH.mkdir(parents=True, exist_ok=True)

print(f"Download path: {DOWNLOAD_PATH.resolve()}")
print(f"Request delay: {REQUEST_DELAY}s")

## Logging Setup

In [ ]:
# Configure logging to file and console
logger = logging.getLogger("scraper")
logger.setLevel(logging.INFO)
logger.handlers.clear()

file_handler = logging.FileHandler("download.log", encoding="utf-8")
file_handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
logger.addHandler(file_handler)

console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter("[%(levelname)s] %(message)s"))
logger.addHandler(console_handler)

logger.info("Scraper initialized")

## Fetch and Parse PDF Links

In [ ]:
session = requests.Session()
session.headers.update({
    "User-Agent": "SC-Decisions-Scraper/1.0 (Academic Research)"
})

logger.info(f"Fetching listing page: {LISTING_URL}")
response = session.get(LISTING_URL, timeout=30)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

# Find all links that point to Philippine Reports PDF files
pdf_links = []
for link in soup.find_all("a", href=True):
    href = link["href"]
    if "philrep_ebooks" in href and href.endswith(".pdf"):
        # Build absolute URL if relative
        if href.startswith("/"):
            url = BASE_URL + href
        elif not href.startswith("http"):
            url = BASE_URL + "/" + href
        else:
            url = href

        # Extract volume title from link text or filename
        title = link.get_text(strip=True)
        if not title:
            title = Path(href).stem  # e.g., Volume_001

        pdf_links.append((title, url))

# Deduplicate by URL while preserving order
seen_urls = set()
unique_links = []
for title, url in pdf_links:
    if url not in seen_urls:
        seen_urls.add(url)
        unique_links.append((title, url))
pdf_links = unique_links

logger.info(f"Found {len(pdf_links)} PDF links")
print(f"Found {len(pdf_links)} PDF links")

# Show first few entries
for title, url in pdf_links[:5]:
    print(f"  {title}: {url}")
if len(pdf_links) > 5:
    print(f"  ... and {len(pdf_links) - 5} more")

## Download PDFs

In [ ]:
downloaded = 0
skipped = 0
errors = 0

for title, url in tqdm(pdf_links, desc="Downloading PDFs"):
    # Derive filename from URL
    filename = Path(url).name
    filepath = DOWNLOAD_PATH / filename

    # Skip if already downloaded
    if filepath.exists() and filepath.stat().st_size > 0:
        logger.debug(f"Skipping (exists): {filename}")
        skipped += 1
        continue

    try:
        logger.info(f"Downloading: {filename}")
        resp = session.get(url, timeout=120, stream=True)
        resp.raise_for_status()

        # Write in chunks to handle large files
        with open(filepath, "wb") as f:
            for chunk in resp.iter_content(chunk_size=8192):
                f.write(chunk)

        size_mb = filepath.stat().st_size / (1024 * 1024)
        logger.info(f"Downloaded: {filename} ({size_mb:.1f} MB)")
        downloaded += 1

    except requests.RequestException as e:
        logger.error(f"Failed to download {filename}: {e}")
        errors += 1
        # Remove partial file
        if filepath.exists():
            filepath.unlink()

    # Polite delay between requests
    time.sleep(REQUEST_DELAY)

## Summary

In [ ]:
summary = (
    f"\n{'='*40}\n"
    f"Download Summary\n"
    f"{'='*40}\n"
    f"Total links found: {len(pdf_links)}\n"
    f"Downloaded:        {downloaded}\n"
    f"Skipped (exist):   {skipped}\n"
    f"Errors:            {errors}\n"
    f"{'='*40}"
)
logger.info(summary)
print(summary)